# Lecture 20 — Implementing LSTMs

**CMU 10-414/714, Deep Learning Systems (lecture by Zico Kolter)**
Companion notebook · Sangram Lembe

Kolter builds an LSTM in plain NumPy and checks it against PyTorch's. PyTorch is not available here, so
each version is checked against an independent reference instead:

1. one LSTM cell, with the four gates computed from **one** big matrix and split
2. a whole sequence, then a batch — and why the batch is stored **time-first**
3. stacked layers, computed layer by layer
4. truncated backpropagation through time, and why passing the hidden state between chunks helps

In [1]:
import numpy as np, time
rng = np.random.default_rng(0)
sigmoid = lambda x: 1 / (1 + np.exp(-x))

## 1. One LSTM cell

PyTorch stores the input weights as a $4d \times n$ matrix and the hidden weights as $4d \times d$ — all
four gates in one block. With input size 20 and hidden size 100 that is $400\times20$ and $400\times100$.
One multiply, then split into four:

In [2]:
def lstm_cell(x, h, c, W_hh, W_ih, b):
    """x: (n,), h, c: (d,). W_hh: (4d, d), W_ih: (4d, n), b: (4d,)"""
    i, f, g, o = np.split(W_hh @ h + W_ih @ x + b, 4)      # ONE matmul, split in four
    i, f, g, o = sigmoid(i), sigmoid(f), np.tanh(g), sigmoid(o)
    c_out = f * c + i * g                                    # the key line
    h_out = o * np.tanh(c_out)
    return h_out, c_out

n, d = 20, 100
W_hh, W_ih, b = rng.normal(0, .1, (4*d, d)), rng.normal(0, .1, (4*d, n)), rng.normal(0, .1, 4*d)
x, h0, c0 = rng.normal(size=n), rng.normal(size=d), rng.normal(size=d)
h1, c1 = lstm_cell(x, h0, c0, W_hh, W_ih, b)
print("W_ih", W_ih.shape, " W_hh", W_hh.shape, " -> h", h1.shape, " c", c1.shape)

W_ih (400, 20)  W_hh (400, 100)  -> h (100,)  c (100,)


**Reference:** the textbook form with eight separate gate matrices. If fusing them into one block is
correct, the two must agree.

In [3]:
def lstm_cell_separate(x, h, c, W_hh, W_ih, b):
    gate = lambda k: W_hh[k*d:(k+1)*d] @ h + W_ih[k*d:(k+1)*d] @ x + b[k*d:(k+1)*d]
    i, f, g, o = sigmoid(gate(0)), sigmoid(gate(1)), np.tanh(gate(2)), sigmoid(gate(3))
    c_out = f * c + i * g
    return o * np.tanh(c_out), c_out

h_ref, c_ref = lstm_cell_separate(x, h0, c0, W_hh, W_ih, b)
print("fused == separate gates:", np.allclose(h1, h_ref) and np.allclose(c1, c_ref))

fused == separate gates: True


Kolter's aside: PyTorch keeps **two** bias vectors, `bias_ih` and `bias_hh`, which are simply added
together. They always receive the same gradient, so one bias does the same job.

## 2. A whole sequence

The full LSTM is the cell applied step after step. Following PyTorch, return every hidden state but
only the **last** cell state.

In [4]:
def lstm(X, h, c, W_hh, W_ih, b):
    H = np.zeros((X.shape[0], d))
    for t in range(X.shape[0]):
        h, c = lstm_cell(X[t], h, c, W_hh, W_ih, b)
        H[t] = h
    return H, c

T = 50
X = rng.normal(size=(T, n))
H, c_last = lstm(X, np.zeros(d), np.zeros(d), W_hh, W_ih, b)
print("all hidden states:", H.shape, "  last cell state:", c_last.shape)

all hidden states: (50, 100)   last cell state: (100,)


### Batching, and why time comes first

Every step multiplies the inputs *at one time step* by the weights. To batch that, all examples at time
$t$ should sit together in memory. Compare the two layouts:

In [5]:
B = 128
X_tbn = rng.normal(size=(T, B, n))                 # time, batch, features  (PyTorch's default)
X_btn = np.ascontiguousarray(X_tbn.transpose(1, 0, 2))   # batch, time, features

print("time-first  X[t]    contiguous:", X_tbn[7].flags["C_CONTIGUOUS"])
print("batch-first X[:, t] contiguous:", X_btn[:, 7].flags["C_CONTIGUOUS"])

def bytes_copied(get):
    """Matmul needs compact input: count bytes that must be copied over a whole sequence."""
    total = 0
    for t in range(T):
        sl = get(t)
        if not sl.flags["C_CONTIGUOUS"]:
            total += sl.nbytes                               # ascontiguousarray would copy this
    return total
print(f"bytes copied per sequence, time-first : {bytes_copied(lambda t: X_tbn[t]):>9,}")
print(f"bytes copied per sequence, batch-first: {bytes_copied(lambda t: X_btn[:, t]):>9,}")

time-first  X[t]    contiguous: True
batch-first X[:, t] contiguous: False
bytes copied per sequence, time-first :         0
bytes copied per sequence, batch-first: 1,024,000


Time-first needs no copies at all; batch-first copies the entire input once per sequence, in 50 scattered
pieces. At this small size the copy is cheap next to the matrix multiply, but it is pure wasted work, and it
is why PyTorch's LSTM defaults to `(time, batch, features)`, unlike convolutions and MLPs. The
batched cell is the same maths with the matrices transposed so the batch dimension leads:

In [6]:
def lstm_cell_batch(x, h, c, W_hh, W_ih, b):
    """x: (B, n), h, c: (B, d)"""
    i, f, g, o = np.split(h @ W_hh.T + x @ W_ih.T + b, 4, axis=1)
    i, f, g, o = sigmoid(i), sigmoid(f), np.tanh(g), sigmoid(o)
    c_out = f * c + i * g
    return o * np.tanh(c_out), c_out

def lstm_batch(X, h, c, W_hh, W_ih, b):
    H = np.zeros((X.shape[0], X.shape[1], d))
    for t in range(X.shape[0]):
        h, c = lstm_cell_batch(X[t], h, c, W_hh, W_ih, b)
        H[t] = h
    return H, c

H_b, _ = lstm_batch(X_tbn, np.zeros((B, d)), np.zeros((B, d)), W_hh, W_ih, b)
H_3, _ = lstm(X_tbn[:, 3], np.zeros(d), np.zeros(d), W_hh, W_ih, b)
print("batched result for example 3 == running example 3 alone:", np.allclose(H_b[:, 3], H_3))

batched result for example 3 == running example 3 alone: True


## 3. Stacking layers

A deep LSTM feeds one layer's hidden states into the next. Because each layer's call processes the whole
sequence, it is natural to compute **layer by layer**. Time step by time step gives the same answer:

In [7]:
layers = [(rng.normal(0, .1, (4*d, d)), rng.normal(0, .1, (4*d, n if k == 0 else d)), np.zeros(4*d))
          for k in range(3)]
Xs = rng.normal(size=(30, n))

def layer_by_layer(X):
    Z = X
    for W_hh_k, W_ih_k, b_k in layers:
        Z, _ = lstm(Z, np.zeros(d), np.zeros(d), W_hh_k, W_ih_k, b_k)
    return Z

def step_by_step(X):
    hs, cs, out = [np.zeros(d)] * 3, [np.zeros(d)] * 3, []
    for t in range(X.shape[0]):
        z = X[t]
        for k, (W_hh_k, W_ih_k, b_k) in enumerate(layers):
            hs[k], cs[k] = lstm_cell(z, hs[k], cs[k], W_hh_k, W_ih_k, b_k)
            z = hs[k]
        out.append(z)
    return np.array(out)

print("layer-by-layer == step-by-step:", np.allclose(layer_by_layer(Xs), step_by_step(Xs)))

layer-by-layer == step-by-step: True


## 4. Truncated backpropagation through time

On a very long sequence you cannot backpropagate through every step at once. Instead, chop it into
chunks and train on each chunk. The question is what hidden state each chunk starts from:

- **reset**: start every chunk from zeros
- **repackage**: start from the previous chunk's last hidden and cell state, *detached* — the value is
  carried over, but no gradient flows back across the boundary

The test: a long stream of bits where the target is the bit from 3 steps earlier, cut into chunks of 6.
At the start of each chunk, the bit to recall is in the *previous* chunk.

In [8]:
import numpy as np

class Tensor:
    """A needle-style tensor: data plus the op and inputs that produced it."""

    def __init__(self, array, requires_grad=True):
        self._init(None, [], np.asarray(array, dtype=np.float64), requires_grad)

    def _init(self, op, inputs, cached, requires_grad):
        self.op, self.inputs = op, list(inputs)
        self.cached, self.requires_grad, self.grad = cached, requires_grad, None

    @classmethod
    def make_const(cls, array):
        """A leaf with no history - what detach() returns."""
        t = Tensor.__new__(Tensor)
        t._init(None, [], array, False)
        return t

    @staticmethod
    def make_from_op(op, inputs):
        out = op.compute(*[i.cached for i in inputs])
        if not any(i.requires_grad for i in inputs):
            return Tensor.make_const(out)        # nothing needs a gradient: no graph
        t = Tensor.__new__(Tensor)
        t._init(op, inputs, out, True)
        return t

    # .data is a detached view that SHARES memory; assigning to it updates in place
    @property
    def data(self):
        return Tensor.make_const(self.cached)
    @data.setter
    def data(self, value):
        self.cached = value.cached if isinstance(value, Tensor) else np.asarray(value)

    def detach(self):  return self.data
    def numpy(self):   return self.cached
    shape = property(lambda s: s.cached.shape)
    def __repr__(self):
        return f"Tensor({np.array2string(self.cached, precision=4)}, op={type(self.op).__name__ if self.op else None})"

    def __add__(s, o):  return EAdd()(s, o) if isinstance(o, Tensor) else AddS(o)(s)
    def __mul__(s, o):  return EMul()(s, o) if isinstance(o, Tensor) else MulS(o)(s)
    def __neg__(s):     return MulS(-1.0)(s)
    def __sub__(s, o):  return s + (-o)
    def __rsub__(s, o): return (-s) + o
    def __pow__(s, c):  return PowS(c)(s)
    def __truediv__(s, o):
        return s * (o ** -1.0) if isinstance(o, Tensor) else MulS(1.0 / o)(s)
    def __matmul__(s, o): return MatMul()(s, o)
    __radd__, __rmul__ = __add__, __mul__

    def backward(self):
        backprop(self, Tensor.make_const(np.ones_like(self.cached)))


class Op:
    def __call__(self, *ins):
        return Tensor.make_from_op(self, ins)
    def grads(self, g, node):
        r = self.gradient(g, node)
        return r if isinstance(r, tuple) else (r,)

class EAdd(Op):
    def compute(s, a, b): return a + b
    def gradient(s, g, n): return g, g
class AddS(Op):
    def __init__(s, c): s.c = c
    def compute(s, a): return a + s.c
    def gradient(s, g, n): return g
class EMul(Op):
    def compute(s, a, b): return a * b
    def gradient(s, g, n): return g * n.inputs[1], g * n.inputs[0]
class MulS(Op):
    def __init__(s, c): s.c = c
    def compute(s, a): return a * s.c
    def gradient(s, g, n): return g * s.c
class PowS(Op):
    def __init__(s, c): s.c = c
    def compute(s, a): return a ** s.c
    def gradient(s, g, n): return g * (n.inputs[0] ** (s.c - 1)) * s.c
class MatMul(Op):
    def compute(s, a, b): return a @ b
    def gradient(s, g, n):
        return g @ transpose(n.inputs[1]), transpose(n.inputs[0]) @ g
class Transpose(Op):
    def compute(s, a): return a.T
    def gradient(s, g, n): return transpose(g)
class Reshape(Op):
    def __init__(s, shape): s.shape = shape
    def compute(s, a): return a.reshape(s.shape)
    def gradient(s, g, n): return reshape(g, n.inputs[0].shape)
class BroadcastTo(Op):
    def __init__(s, shape): s.shape = shape
    def compute(s, a): return np.broadcast_to(a, s.shape).copy()
    def gradient(s, g, n):                        # broadcast forward = sum backward
        ins = n.inputs[0].shape
        pad = len(s.shape) - len(ins)
        axes = tuple(range(pad)) + tuple(pad + i for i, d in enumerate(ins)
                                         if d == 1 and s.shape[pad + i] != 1)
        return reshape(summation(g, axes) if axes else g, ins)
class Summation(Op):
    def __init__(s, axes=None):
        s.axes = (axes,) if isinstance(axes, int) else axes
    def compute(s, a): return a.sum(axis=s.axes)
    def gradient(s, g, n):
        ins = n.inputs[0].shape
        ax = range(len(ins)) if s.axes is None else s.axes
        keep = tuple(1 if i in ax else d for i, d in enumerate(ins))
        return broadcast_to(reshape(g, keep), ins)
class ReLUOp(Op):
    def compute(s, a): return np.maximum(0, a)
    def gradient(s, g, n):
        return g * Tensor.make_const((n.inputs[0].cached > 0) * 1.0)
class Exp(Op):
    def compute(s, a): return np.exp(a)
    def gradient(s, g, n): return g * exp(n.inputs[0])
class LogSumExp(Op):
    """Stable log-sum-exp over axis 1: subtract the row max first."""
    def compute(s, a):
        m = a.max(1, keepdims=True)
        return (np.log(np.exp(a - m).sum(1, keepdims=True)) + m)[:, 0]
    def gradient(s, g, n):
        z = n.inputs[0]
        sm = exp(z - broadcast_to(reshape(n, (z.shape[0], 1)), z.shape))
        return broadcast_to(reshape(g, (z.shape[0], 1)), z.shape) * sm

transpose    = lambda a: Transpose()(a)
reshape      = lambda a, s: Reshape(s)(a)
broadcast_to = lambda a, s: BroadcastTo(s)(a)
summation    = lambda a, axes=None: Summation(axes)(a)
relu         = lambda a: ReLUOp()(a)
exp          = lambda a: Exp()(a)
logsumexp    = lambda a: LogSumExp()(a)

def topo_sort(root):
    """Post-order DFS, iterative so that very deep graphs cannot overflow the stack."""
    order, seen, stack = [], set(), [(root, False)]
    while stack:
        node, done = stack.pop()
        if done:
            order.append(node)
            continue
        if id(node) in seen:
            continue
        seen.add(id(node))
        stack.append((node, True))
        for i in node.inputs:
            if id(i) not in seen:
                stack.append((i, False))
    return order

def backprop(out, seed):
    """Reverse-mode AD: walk reverse topological order, summing partial adjoints."""
    parts = {id(out): [seed]}
    for node in reversed(topo_sort(out)):
        if id(node) not in parts: continue
        g = parts[id(node)][0]
        for extra in parts[id(node)][1:]:
            g = g + extra
        node.grad = g
        if node.op is not None:
            for inp, p in zip(node.inputs, node.op.grads(g, node)):
                if inp.requires_grad:
                    parts.setdefault(id(inp), []).append(p)

def numeric_grad(f, param, eps=1e-6):
    """Two-sided finite differences of scalar f() w.r.t. every entry of param."""
    g = np.zeros_like(param.cached)
    for idx in np.ndindex(*param.cached.shape):
        old = param.cached[idx]
        param.cached[idx] = old + eps; a = float(f().cached)
        param.cached[idx] = old - eps; b = float(f().cached)
        param.cached[idx] = old
        g[idx] = (a - b) / (2 * eps)
    return g



class Tanh(Op):
    def compute(s, a): return np.tanh(a)
    def gradient(s, g, n): return g * (1.0 - n * n)          # d tanh = 1 - tanh^2
class Sigmoid(Op):
    def compute(s, a): return 1.0 / (1.0 + np.exp(-a))
    def gradient(s, g, n): return g * n * (1.0 - n)          # d sigma = sigma (1 - sigma)
class Slice(Op):
    """Take columns [a:b] of a 2-D tensor - used to split the LSTM's 4d output."""
    def __init__(s, a, b): s.a, s.b = a, b
    def compute(s, x): return x[:, s.a:s.b]
    def gradient(s, g, n):
        full = np.zeros(n.inputs[0].shape); full[:, s.a:s.b] = 1.0
        pad = PadCols(s.a, n.inputs[0].shape[1])
        return pad(g)
class PadCols(Op):
    def __init__(s, a, total): s.a, s.total = a, total
    def compute(s, g):
        out = np.zeros((g.shape[0], s.total)); out[:, s.a:s.a + g.shape[1]] = g; return out
    def gradient(s, g, n): return Slice(s.a, s.a + n.inputs[0].shape[1])(g)
tanh    = lambda a: Tanh()(a)
sigmoid = lambda a: Sigmoid()(a)
cols    = lambda a, i, j: Slice(i, j)(a)

def param(rng, *shape, scale=None):
    scale = scale if scale is not None else 1.0 / np.sqrt(shape[0])
    return Tensor(rng.uniform(-scale, scale, shape))

class RNNCell:
    def __init__(self, n, d, rng):
        self.Wx, self.Wh, self.b = param(rng, n, d), param(rng, d, d), Tensor(np.zeros((1, d)))
    def params(self): return [self.Wx, self.Wh, self.b]
    def __call__(self, x, h):
        z = x @ self.Wx + h @ self.Wh
        return tanh(z + broadcast_to(self.b, z.shape))

class LSTMCell:
    def __init__(self, n, d, rng, forget_bias=1.0):
        self.d = d
        self.Wx, self.Wh = param(rng, n, 4 * d), param(rng, d, 4 * d)     # ONE big matrix each
        b = np.zeros((1, 4 * d)); b[0, d:2 * d] = forget_bias              # common trick: start f near 1
        self.b = Tensor(b)
    def params(self): return [self.Wx, self.Wh, self.b]
    def __call__(self, x, state):
        h, c = state
        d = self.d
        z = x @ self.Wx + h @ self.Wh
        z = z + broadcast_to(self.b, z.shape)
        i, f = sigmoid(cols(z, 0, d)), sigmoid(cols(z, d, 2 * d))
        g, o = tanh(cols(z, 2 * d, 3 * d)), sigmoid(cols(z, 3 * d, 4 * d))
        c = c * f + i * g                                                  # the key line
        h = tanh(c) * o
        return h, c

class AdamOpt:
    def __init__(self, params, lr=0.01):
        self.p, self.lr, self.t = params, lr, 0
        self.m = [np.zeros_like(q.cached) for q in params]; self.v = [np.zeros_like(q.cached) for q in params]
    def step(self):
        self.t += 1
        for k, q in enumerate(self.p):
            if q.grad is None: continue
            g = q.grad.cached
            self.m[k] = 0.9 * self.m[k] + 0.1 * g; self.v[k] = 0.999 * self.v[k] + 0.001 * g * g
            q.data = q.cached - self.lr * (self.m[k] / (1 - 0.9 ** self.t)) / (np.sqrt(self.v[k] / (1 - 0.999 ** self.t)) + 1e-8)
            q.grad = None


def train_tbptt(repackage, epochs=12, d=16, chunk=6, delay=3, B=32, T=240, seed=0):
    rng = np.random.default_rng(seed)
    cell = LSTMCell(1, d, rng)
    Wy, by = param(rng, d, 2), Tensor(np.zeros((1, 2)))
    opt = AdamOpt(cell.params() + [Wy, by], 0.02)
    graph_sizes = []

    def run(x, y, train):
        h, c = Tensor.make_const(np.zeros((B, d))), Tensor.make_const(np.zeros((B, d)))
        correct = total = 0
        for s in range(0, T, chunk):                      # one chunk at a time
            if not repackage:
                h, c = Tensor.make_const(np.zeros((B, d))), Tensor.make_const(np.zeros((B, d)))
            loss = None
            for t in range(s, s + chunk):
                h, c = cell(Tensor.make_const(x[:, t:t+1]), (h, c))
                if t >= delay:
                    logits = h @ Wy
                    logits = logits + broadcast_to(by, logits.shape)
                    Y = Tensor.make_const(np.eye(2)[y[:, t]])
                    l = summation(logsumexp(logits) - summation(logits * Y, 1)) / B
                    loss = l if loss is None else loss + l
                    correct += (logits.cached.argmax(1) == y[:, t]).sum(); total += B
            if train and loss is not None:
                loss.backward(); opt.step()
                graph_sizes.append(len(topo_sort(loss)))
            h, c = h.detach(), c.detach()                 # keep the VALUE, drop the graph
        return correct / total

    for e in range(epochs):
        x = rng.integers(0, 2, (B, T)).astype(float)
        y = np.zeros((B, T), dtype=int); y[:, delay:] = x[:, :-delay]
        run(x, y, train=True)
    x = np.random.default_rng(99).integers(0, 2, (B, T)).astype(float)
    y = np.zeros((B, T), dtype=int); y[:, delay:] = x[:, :-delay]
    return run(x, y, train=False), max(graph_sizes)

acc_reset, g1 = train_tbptt(repackage=False)
acc_repack, g2 = train_tbptt(repackage=True)
print(f"reset to zeros each chunk : accuracy {acc_reset:.3f}")
print(f"repackaged hidden state   : accuracy {acc_repack:.3f}")
print(f"largest graph backpropagated through: {g1} vs {g2} nodes (bounded by the chunk either way)")

reset to zeros each chunk : accuracy 0.748
repackaged hidden state   : accuracy 1.000
largest graph backpropagated through: 192 vs 192 nodes (bounded by the chunk either way)


With resets, the first three predictions of each chunk are guesses — the bit they need was in the
previous chunk — which caps accuracy near $\tfrac12\cdot1 + \tfrac12\cdot\tfrac12 = 0.75$. Repackaging carries
the memory across the boundary, and accuracy climbs well past that. Because the carried state is
detached, the graph never grows beyond one chunk. That is why PyTorch's LSTM returns the last hidden
**and** cell state: so you can detach them and pass them on.

## Summary

- One $4d$-wide matrix multiply, split into four gates, equals eight separate gate matrices.
- Time-first layout keeps every time step's batch contiguous; batch-first forces a copy at each step.
- The batched LSTM matched each example run alone, and layer-by-layer stacking matched step-by-step.
- Truncated BPTT with repackaged, detached state beat resetting each chunk, with a bounded graph.